In [ ]:
# LLDA

#https://ujangriswanto08.medium.com/step-by-step-implementation-of-labeled-lda-in-python-or-r-bbc9b1e09958
# executed with conda environment python 3.12.7

#%pip install tomotopy
#%pip install nltk spacy

import tomotopy as tp
import pandas as pd
import numpy as np
import sklearn 
from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
import spacy
import matplotlib.pyplot as plt
import seaborn as sns
import re
import unicodedata


In [ ]:
# load data
data = pd.read_csv("reviews/df_merge_openalexscopus_checked.csv",encoding="latin1")

data_f = data.dropna(subset= ["top_1", "top_2", "text_preprocessed"])

# prepare data subset for LLDA
data_f = pd.DataFrame({
    "Document": data_f["text_preprocessed"],
    "Labels": data_f[["top_1", "top_2"]]
        .apply(lambda x: ", ".join(x.dropna().astype(str)), axis=1)
})

In [ ]:
# count the occurrences of each label in data_f
label_counts = data_f['Labels'].value_counts()


In [ ]:
# -----------------------------
# Step 1: Build label map
# -----------------------------
all_labels = data_f['Labels'].str.split(',\s*').explode().unique()
all_labels = sorted([label.strip() for label in all_labels])
label_map = {label: idx for idx, label in enumerate(all_labels)}

#all_labels = (data_f['Labels'].str.split(',').explode().str.strip().unique())
#all_labels = [str(lbl) for lbl in all_labels]
#label_map = {lbl: lbl for lbl in all_labels}

num_topics = len(label_map)
print("Label map:", label_map)

In [ ]:
# -----------------------------
# Step 2: Build doc_labels
# -----------------------------
doc_labels = []
for lbls in data_f['Labels']:
    labels_clean = [l.strip() for l in lbls.split(',') if l.strip() != ""]
    # Ensure each label is a plain Python int
    doc_labels.append([int(label_map[l]) for l in labels_clean])


In [ ]:
# -----------------------------
# Step 3: Tokenize documents (safe ngrams) and clean them
# -----------------------------

def clean_token(token):
    """
    Force token to be pure Python str, remove accents, emojis, control characters, 
    and non-alphanumeric symbols except underscore.
    """
    # Force Python str
    token = str(token).strip()
    if not token:
        return None

    # Normalize unicode (NFKD) and remove accents
    token = unicodedata.normalize('NFKD', token)
    token = "".join([c for c in token if not unicodedata.combining(c)])

    # Remove non-ASCII characters
    token = token.encode('ascii', errors='ignore').decode('ascii')

    # Keep only letters, digits, underscores
    token = re.sub(r'[^\w\d_]', '', token)

    # Remove empty tokens
    if not token:
        return None

    return token

X_text = []
for doc in data_f['Document']:
    tokens = [clean_token(t) for t in doc.split()]
    tokens = [t for t in tokens if t]  # remove empty tokens
    X_text.append(tokens)


In [ ]:
# Optional: sanity check
bad_tokens_remaining = sum([len(toks) for toks in X_text if not toks])

In [ ]:
# -----------------------------
# Step 4: Initialize LLDA
# -----------------------------
model = tp.LLDAModel(
    k=num_topics,    # number of unique labels
    tw=tp.TermWeight.ONE,
    alpha=0.1,
    eta=0.01,
    seed=42
)

In [131]:
# -----------------------------
# Step 5: Add documents
# -----------------------------
for doc_tokens, labels in zip(X_text, doc_labels):
    # labels must be Python ints
    labels = [str(l) for l in labels]  # <- this ensures pure Python ints
    #doc_tokens = [str(z) for z in X_text]
    model.add_doc(doc_tokens, labels=labels)

In [ ]:
# -----------------------------
# Step 6: Train the model
# -----------------------------
for i in range(0, 500, 50):  # 500 iterations in steps of 50
    model.train(50)
    print(f"Iteration {i+50}, log-likelihood per word: {model.ll_per_word:.4f}")


In [ ]:
# -----------------------------
# Step 7: Inspect top words per label/topic
# -----------------------------
for label, idx in label_map.items():
    print(f"\nLabel: {label}")
    top_words = model.get_topic_words(idx, top_n=10)
    print([w for w, _ in top_words])